# PCA Experiments

## Objective

Investigate whether Principal Component Analysis (PCA) improves the classification of schizophrenia, bipolar disorder, and healthy controls using gene-expression data.

## Workflow

1. Load the cleaned expression matrix and sample labels.
2. Create a stratified training/test split.
3. Compare logistic regression with and without PCA.
4. Evaluate PCA with different numbers of components using cross-validation.
5. Compare PCA-based logistic regression, K-nearest neighbours (KNN), and random forest.
6. Evaluate selected configurations and compare their results with the baseline.

Imputation, scaling, and PCA are fitted within each cross-validation fold to reduce data leakage. The test set is kept separate from the cross-validation process.

The dataset contains many more gene-expression features than samples. PCA may help reduce dimensionality, but its effect on classification performance must be assessed empirically.

## Import Libraries and Set the Random Seed
Import the libraries required for data handling, preprocessing, PCA, model training, cross-validation, and performance evaluation. A fixed random seed is used to make the experiments reproducible.

In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

RANDOM_STATE = 42

PCA notebook imports successful.


## Load the Expression Matrix and Metadata

In [2]:
PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

X = pd.read_csv(
    PROCESSED_DIR / "expression_matrix.csv",
    index_col=0,
)

metadata = pd.read_csv(
    PROCESSED_DIR / "sample_metadata.csv",
    index_col=0,
)

X.index = X.index.astype(str)
metadata.index = metadata.index.astype(str)

if "sample_id" in metadata.columns:
    metadata = metadata.set_index("sample_id")

metadata.index = metadata.index.astype(str)
metadata = metadata.loc[X.index]

y = metadata["disease state"]

assert X.index.equals(y.index)
assert X.index.is_unique
assert y.notna().all()

print("Expression matrix:", X.shape)
print("Number of samples:", len(y))
print("\nClass distribution:")
print(y.value_counts())

Expression matrix: (88, 30635)
Metadata: (88, 8)
Metadata columns: ['title', 'tissue', 'disease state', 'age', 'population', 'dsm-iv', 'age of onset', 'gender']


## Create a Stratified Training/Test Split
A stratified split preserves the approximate class proportions in the training and test sets. Cross-validation and model selection use the training set; the test set is reserved for evaluation.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert set(X_train.index).isdisjoint(X_test.index)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))
print("\nTraining class counts:")
print(y_train.value_counts())
print("\nTest class counts:")
print(y_test.value_counts())

Training samples: 69
Test samples: 18

Training class counts:
disease state
bipolar disorder    23
schizophrenia       23
healthy control     23
Name: count, dtype: int64

Test class counts:
disease state
schizophrenia       6
healthy control     6
bipolar disorder    6
Name: count, dtype: int64


## Compare PCA Component Counts Across Models
Each classifier is evaluated with several PCA component counts using five-fold stratified cross-validation on the training set. The same folds and macro-F1 metric are used for each configuration. Imputation, scaling, and PCA are fitted separately within each fold.

In [12]:
models = {
    "Logistic regression": LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=2000,
        random_state=RANDOM_STATE,
    ),
    "KNN": KNeighborsClassifier(
        n_neighbors=9,
        weights="distance",
    ),
    "Random forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
}

component_options = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]

pca_results = []

for model_name, classifier in models.items():
    for n_components in component_options:
        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=n_components)),
            ("classifier", clone(classifier)),
        ])

        scores = cross_val_score(
            pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring="f1_macro",
            n_jobs=-1,
            error_score="raise",
        )

        pca_results.append({
            "Model": model_name,
            "Components": n_components,
            "Mean CV macro F1": scores.mean(),
            "Std CV macro F1": scores.std(),
        })

pca_results_df = pd.DataFrame(pca_results)

display(
    pca_results_df
    .sort_values("Mean CV macro F1", ascending=False)
    .round(3)
    .reset_index(drop=True)
)

,Model,Components,Mean CV macro F1,Std CV macro F1
0,Logistic regression,50,0.419,0.047
1,Random forest,45,0.413,0.096
2,Logistic regression,15,0.406,0.076
3,Logistic regression,40,0.382,0.062
4,Logistic regression,20,0.364,0.107
5,Random forest,40,0.357,0.025
6,Random forest,25,0.352,0.100
7,Logistic regression,45,0.349,0.060
8,Logistic regression,30,0.345,0.087
9,Random forest,50,0.343,0.071


## Identify the best component count for each model

In [13]:

best_pca_results = (
    pca_results_df
    .sort_values("Mean CV macro F1", ascending=False)
    .groupby("Model")
    .first()
    .reset_index()
    .sort_values("Mean CV macro F1", ascending=False)
)

display(best_pca_results.round(3))


,Model,Components,Mean CV macro F1,Std CV macro F1
1,Logistic regression,50,0.419,0.047
2,Random forest,45,0.413,0.096
0,KNN,50,0.340,0.107


## Compare PCA Against the Original Features
This experiment compares each model's cross-validation performance using the original gene-expression features against its best-performing PCA configuration. Both methods use the same training data and cross-validation folds.

In [14]:
comparison_results = []

# First: evaluate each model WITHOUT PCA
for model_name, classifier in models.items():

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", clone(classifier))
    ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1
    )

    comparison_results.append({
        "Model": model_name,
        "Method": "No PCA",
        "Components": "N/A",
        "Mean CV macro F1": scores.mean(),
        "Std CV macro F1": scores.std()
    })

# Then: add the best PCA result for each model
for _, row in best_pca_results.iterrows():

    comparison_results.append({
        "Model": row["Model"],
        "Method": "PCA",
        "Components": int(row["Components"]),
        "Mean CV macro F1": row["Mean CV macro F1"],
        "Std CV macro F1": row["Std CV macro F1"]
    })

comparison_df = pd.DataFrame(comparison_results)

display(
    comparison_df
    .sort_values("Mean CV macro F1", ascending=False)
    .round(3)
    .reset_index(drop=True)
)

,Model,Method,Components,Mean CV macro F1,Std CV macro F1
0,Logistic regression,PCA,50,0.419,0.047
1,Random forest,PCA,45,0.413,0.096
2,Logistic regression,No PCA,N/A,0.398,0.077
3,KNN,PCA,50,0.340,0.107
4,KNN,No PCA,N/A,0.338,0.117
5,Random forest,No PCA,N/A,0.332,0.061


## Evaluate the Selected PCA Models
For each classifier, select the component count with the highest mean cross-validation macro F1. Fit the resulting pipeline on the training set and evaluate it on the held-out test set. Do not select component counts based on test-set performance.

In [15]:
best_components = {
    row["Model"]: int(row["Components"])
    for _, row in best_pca_results.iterrows()
}

final_test_results = []
final_pca_models = {}
final_pca_predictions = {}

for model_name, n_components in best_components.items():
    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_components)),
        ("classifier", clone(models[model_name])),
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    final_pca_models[model_name] = pipeline
    final_pca_predictions[model_name] = y_pred

    final_test_results.append({
        "Model": model_name,
        "PCA components": n_components,
        "Test accuracy": accuracy_score(y_test, y_pred),
        "Test macro F1": f1_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0,
        ),
    })

final_test_results_df = pd.DataFrame(final_test_results)

display(
    final_test_results_df
    .sort_values("Test macro F1", ascending=False)
    .round(3)
    .reset_index(drop=True)
)


Logistic regression + PCA (50 components)
                  precision    recall  f1-score   support

bipolar disorder       0.60      0.50      0.55         6
 healthy control       0.44      0.67      0.53         6
   schizophrenia       0.25      0.17      0.20         6

        accuracy                           0.44        18
       macro avg       0.43      0.44      0.43        18
    weighted avg       0.43      0.44      0.43        18


Random forest + PCA (45 components)
                  precision    recall  f1-score   support

bipolar disorder       0.00      0.00      0.00         6
 healthy control       0.50      0.67      0.57         6
   schizophrenia       0.38      0.50      0.43         6

        accuracy                           0.39        18
       macro avg       0.29      0.39      0.33        18
    weighted avg       0.29      0.39      0.33        18


KNN + PCA (50 components)
                  precision    recall  f1-score   support

bipolar disorder

,Model,PCA components,Test accuracy,Test macro F1
0,Logistic regression,50,0.444,0.426
1,KNN,50,0.389,0.371
2,Random forest,45,0.389,0.333


In [16]:
# Evaluate a majority-class baseline on the same split.
dummy_model = DummyClassifier(strategy="most_frequent")

dummy_cv_scores = cross_val_score(
    dummy_model,
    X_train,
    y_train,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
)

dummy_model.fit(X_train, y_train)
dummy_predictions = dummy_model.predict(X_test)

dummy_results = {
    "Model": "DummyClassifier",
    "Method": "Baseline",
    "Components": "N/A",
    "CV macro F1": dummy_cv_scores.mean(),
    "CV F1 std": dummy_cv_scores.std(),
    "Test accuracy": accuracy_score(y_test, dummy_predictions),
    "Test macro F1": f1_score(
        y_test,
        dummy_predictions,
        average="macro",
        zero_division=0,
    ),
}

# Combine the selected PCA models' actual results.
final_comparison = final_test_results_df.merge(
    best_pca_results[
        ["Model", "Components", "Mean CV macro F1", "Std CV macro F1"]
    ],
    on="Model",
    how="left",
)

final_comparison = final_comparison.rename(columns={
    "PCA components": "Components",
    "Mean CV macro F1": "CV macro F1",
    "Std CV macro F1": "CV F1 std",
    "Test accuracy": "Test accuracy",
    "Test macro F1": "Test macro F1",
})

final_comparison["Method"] = "PCA"

final_comparison = pd.concat(
    [
        final_comparison,
        pd.DataFrame([dummy_results]),
    ],
    ignore_index=True,
)

display(
    final_comparison
    .sort_values("Test macro F1", ascending=False)
    .round(3)
    .reset_index(drop=True)
)

,Model,Method,Components,CV macro F1,CV F1 std,Test accuracy,Test macro F1
0,Logistic regression,PCA,50,0.419,0.047,0.444,0.426
1,Random forest,PCA,45,0.413,0.096,0.389,0.333
2,KNN,PCA,50,0.340,0.107,0.389,0.371


## Results and Interpretation
The cross-validation results identify the best-performing PCA component count for each classifier. The final comparison reports the corresponding held-out test accuracy and macro F1-score, alongside the DummyClassifier baseline.

Macro F1-score gives equal weight to each class, making it useful for assessing multiclass performance. The cross-validation standard deviation indicates how much performance varies across the validation folds.

The test set contains only 18 samples, so a small number of different predictions can noticeably change the reported metrics. The results should therefore be interpreted cautiously and do not establish reliable diagnostic discrimination.

PCA reduces the number of input features, but it does not guarantee better classification. Its value should be judged by comparing cross-validation performance with and without PCA, while keeping model selection separate from test-set evaluation.

Logistic regression with 50 PCA components achieved the highest held-out test macro-F1 score (0.426) and test accuracy (0.444) among the evaluated models. K-nearest neighbours achieved a test macro-F1 of 0.371, while random forest achieved 0.333. All three PCA-based models obtained higher test macro-F1 scores than the DummyClassifier baseline (0.167).